# Тетрадь 1.1: Парсеры сайтов и данных

В тетради 1.0 мы собрали `policy.json` руками — 50 правил из открытых источников. Теперь автоматизируем этот процесс.

**Чему ты научишься:**

- отправлять HTTP-запросы через `requests`;
- парсить HTML через `BeautifulSoup`;
- работать с публичными API;
- решать проблему сертификатов для российских источников;
- писать парсеры, которые не ломаются при изменении сайта.

**Что на выходе:**

- Модуль `src/underwriting/parsers.py` с функциями `fetch_html` и `extract_rules_from_html`.
- Тесты в `tests/test_parsers.py`.
- 5–10 новых правил для `policy.json`, собранных автоматически.

**Пункты чеклиста:**

- №11: парсеры сайтов и данных.
- №4: работа с API, внутренними сервисами, источниками данных.
- №15: SQL, HTTP, REST API, клиент-серверная архитектура.
- №9 (частично): логирование и обработка ошибок.

**Формат работы:**

Каждая функция — отдельный раздел. Внутри: пример использования (исполняемая ячейка с комментариями), затем 1–2 аналогичные задачи для самостоятельного решения. Решения не даются.

## 0. Связь с предыдущими тетрадями

В фазе 0 мы построили окружение и модули `credit_math.py`, `schemas.py`, `generate.py`. В тетради 1.0 создали `policy.json` — 50 правил кредитной политики, собранных вручную.

**Проблема ручного сбора:** банки меняют условия, ЦБ выпускает новые нормативы. Перепроверять 50 правил каждые три месяца — это недели работы. Нужен автоматический парсер.

**Что делаем здесь:** пишем парсер, который скачивает страницы банков, ищет текст, похожий на правила, и возвращает кандидатов. Дальше (в фазе 2) эти кандидаты пойдут в LLM, которая превратит их в структурированные правила формата из `policy.json`.

**Важно:** парсер не решает, является ли текст правилом. Он находит кандидатов. Решение принимает человек или LLM.

In [25]:
# Импорты из фазы 0.
from pathlib import Path
import json

from underwriting.policy import load_policy

policy = load_policy()
print(f"Загружено правил из policy.json: {len(policy)}")

Загружено правил из policy.json: 50


## 1. Функция `fetch_html`: скачивание HTML по URL

**Что делает:** отправляет GET-запрос на указанный URL, возвращает HTML-строку. При ошибке возвращает пустую строку и логирует проблему.

**Зачем нужна:** это база для любого парсинга. Сначала скачать, потом разбирать.

In [26]:
# ПРИМЕР ИСПОЛЬЗОВАНИЯ fetch_html
# Простой случай: скачиваем страницу с сайта, созданного для обучения парсингу.
import requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

URL_SAFE = "https://quotes.toscrape.com/"

response = requests.get(URL_SAFE, headers=HEADERS, timeout=10)
print(f"Статус: {response.status_code}")
print(f"Размер: {len(response.text)} символов")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print()
print("Первые 300 символов:")
print(response.text[:300])

Статус: 200
Размер: 11021 символов
Content-Type: text/html; charset=utf-8

Первые 300 символов:
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">



### Задача 1.1: используй `fetch_html` на другом безопасном сайте

Открой любой другой сайт, созданный для обучения парсингу (например, `http://books.toscrape.com/` или `https://httpbin.org/html`). Отправь запрос, выведи статус и первые 200 символов ответа.

**Условие:** используй тот же подход с `HEADERS` и `timeout`.

In [31]:
response = requests.get('http://books.toscrape.com/', headers=HEADERS, timeout=10)
print(response.status_code, response.text[400:500])

200 ape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


### Задача 1.2: обработай ошибку

Отправь запрос на заведомо несуществующий домен (`https://this-domain-does-not-exist-12345.ru`). Оберни вызов в `try/except`, поймай ошибку `requests.exceptions.RequestException` и выведи сообщение «Сайт недоступен».

**Подсказка:** `requests.get` бросает `ConnectionError`, `Timeout`, `SSLError` — все они наследуются от `RequestException`.

In [33]:
try:
    response = requests.get('https://this-domain-does-not-exist-12345.ru', headers=HEADERS, timeout=10)
except requests.exceptions.RequestException:
    print('Сайт недоступен')

Сайт недоступен


## 2. Функция `fetch_html` для российских сайтов (проблема SSL)

**Что происходит:** российские банки подписаны корневым сертификатом Минцифры (`Russian Trusted Sub CA`). Этот сертификат не входит в стандартную связку `certifi`, которую использует Python. Поэтому `requests` падает с `SSLCertVerificationError`.

**Решения:**

1. Скачать `russiantrustedca.pem` и добавить в `certifi` — правильно для продакшна.
2. Прописать `REQUESTS_CA_BUNDLE` — правильно для локальной разработки.
3. Отключить проверку (`verify=False`) — временно для тетради.

In [43]:
# ПРИМЕР: запрос к Сбербанку без фикса SSL
URL_SBER = "https://www.vtb.ru/personal/kredit/kalkulyator/"

try:
    r = requests.get(URL_SBER, headers=HEADERS, timeout=10)
    print(f"Статус: {r.status_code}")
except requests.exceptions.SSLError as e:
    print("ОШИБКА SSL:")
    print(str(e)[:300])

Статус: 200


## 3. Функция `extract_rules_from_html`: поиск кандидатов в правила

**Что делает:** принимает HTML-строку и URL источника, находит текст, похожий на правила кредитной политики (абзацы и элементы списков с ключевыми словами), возвращает список кандидатов.

**Зачем нужна:** банки не публикуют правила в структурированном виде. Они пишут их прозой на страницах. Нужно вытащить эти фразы и передать дальше (в LLM).

In [49]:
# ПРИМЕР ИСПОЛЬЗОВАНИЯ extract_rules_from_html
from bs4 import BeautifulSoup

def extract_candidates(html: str, source_url: str) -> list[dict]:
    """Найти кандидатов в правила политики в HTML."""
    if not html:
        return []

    soup = BeautifulSoup(html, "lxml")
    keywords = ["возраст", "доход", "стаж", "просрочк", "DTI", "PTI", "руб", "%"]
    found: list[dict] = []
    seen: set[str] = set()

    for tag in soup.find_all(["p", "li", "h3", "h4",'span']):
        text = tag.get_text(strip=True)
        if not text or len(text) < 15 or len(text) > 400:
            continue
        if text in seen:
            continue
        if any(k.lower() in text.lower() for k in keywords):
            seen.add(text)
            found.append({"source_url": source_url, "raw_text": text})

    return found

# Используем на HTML Сбербанка
candidates = extract_candidates(r.text, URL_SBER)
print(f"Найдено кандидатов: {len(candidates)}")
for c in candidates[:5]:
    print(f"• {c['raw_text'][:120]}")

Найдено кандидатов: 6
• возраст заемщика не соответствует требованиям банка
• нет официального дохода или его размера недостаточно для получения запрошенной суммы
• Возраст от 23 лет на момент подачи заявки до 75 лет на момент погашения кредита
• Официальный доход от 10 тыс. ₽ в месяц
• Общий трудовой стаж от 1 года


### Задача 3.1: прогони `extract_rules_from_html` на ВТБ

Скачай страницу `https://www.vtb.ru/personal/kredit/kalkulyator/` (с `verify=False`), прогони через `extract_candidates`, выведи найденных кандидатов.

**Условие:** используй функцию `fetch_html` из модуля `parsers.py` (создай его по ходу), а не пиши запрос заново.

In [39]:
# Используем на HTML Сбербанка
candidates = extract_candidates(r.text, 'https://www.vtb.ru/personal/kredit/kalkulyator/')
print(f"Найдено кандидатов: {len(candidates)}")
for c in candidates[:5]:
    print(f"• {c['raw_text'][:120]}")

Найдено кандидатов: 0


### Задача 3.2: расширь список тегов

В `extract_candidates` сейчас ищутся теги `p`, `li`, `h3`, `h4`. Добавь `span` и `div`. Проверь, стало ли кандидатов больше на той же странице ВТБ.

**Подсказка:** в `soup.find_all` можно передать список тегов: `["p", "li", "h3", "h4", "span", "div"]`.

In [46]:
def extract_candidates(html: str, source_url: str) -> list[dict]:
    """Найти кандидатов в правила политики в HTML."""
    if not html:
        return []

    soup = BeautifulSoup(html, "lxml")
    keywords = ["возраст", "доход", "стаж", "просрочк", "DTI", "PTI", "руб", "%", 'span', 'div', 'под']
    found: list[dict] = []
    seen: set[str] = set()

    for tag in soup.find_all(["p", "li", "h3", "h4", 'span', 'div']):
        text = tag.get_text(strip=True)
        if not text or len(text) < 15 or len(text) > 400:
            continue
        if text in seen:
            continue
        if any(k.lower() in text.lower() for k in keywords):
            seen.add(text)
            found.append({"source_url": source_url, "raw_text": text})

    return found

# Используем на HTML Сбербанка
r.encoding = "utf-8"
candidates = extract_candidates(r.text, URL_SBER)
print(f"Найдено кандидатов: {len(candidates)}")
for c in candidates[:5]:
    print(f"• {c['raw_text'][:120]}")

Найдено кандидатов: 36
• Кредит наличными под залог автомобиляСтавка по кредиту с залогом будет ниже до 5%, чем по кредиту без залога, а одобренн
• Кредит наличными под залог автомобиляСтавка по кредиту с залогом будет ниже до 5%, чем по кредиту без залога, а одобренн
• Кредит наличными под залог автомобиля
• Ставка по кредиту с залогом будет ниже до 5%, чем по кредиту без залога, а одобренная сумма больше
• ОформитьПодробнее


## 4. Модуль `src/underwriting/parsers.py`

Код в тетради — черновой. Продовая версия живёт в модуле и покрыта тестами.

Создай `src/underwriting/parsers.py` со следующими функциями:

- `fetch_html(url, timeout=10, verify=True) -> str` — скачивает HTML. Возвращает пустую строку при ошибке. Логирует через `logger.warning`.
- `extract_rules_from_html(html, source_url) -> list[ParsedRule]` — находит кандидатов. Возвращает список объектов `ParsedRule` (dataclass с полями `source_url` и `raw_text`).
- `extract_rules_from_urls(urls, verify=True) -> list[ParsedRule]` — высокоуровневая обёртка: проходит по списку URL, для каждого вызывает `fetch_html` и `extract_rules_from_html`, объединяет результаты.

**Дополнительно:**

- Добавь retry в `fetch_html`: 3 попытки с паузой 1–2 секунды.
- Логируй каждый неудачный запрос: URL и причину.

### Задача 4.1: создай модуль `parsers.py`

Перенеси логику из тетради в `src/underwriting/parsers.py`. Создай dataclass `ParsedRule`, напиши три функции.

**Условие:** все функции должны иметь type hints и docstrings.

In [ ]:
# Проверь, что модуль импортируется и работает.
# from underwriting.parsers import fetch_html, extract_rules_from_urls
# rules = extract_rules_from_urls([URL_SBER], verify=False)
# print(f"Найдено: {len(rules)}")

## 5. Сбор кандидатов с нескольких источников

Собери кандидатов с 3–4 сайтов, объедини в один список, сохрани в `data/parsed_candidates.json`.

In [50]:
from underwriting.parsers import extract_rules_from_urls
print(extract_rules_from_urls(['https://alfa-gid.vercel.app/articles/k/kr/kreditnaya-politika-ao-alfa-bank.html']))

[ParsedRule(source_url='https://alfa-gid.vercel.app/articles/k/kr/kreditnaya-politika-ao-alfa-bank.html', raw_text='Основной целью банка является баланс между прибыльностью портфеля и минимизацией невозвратов. Именно поэтомуоценка кредитоспособностипроводится всесторонне, охватывая не только текущие доходы, но и историю поведения клиента в других финансовых институтах. Знание этих нюансов позволяет избежать типичных ошибок при подаче заявки.'), ParsedRule(source_url='https://alfa-gid.vercel.app/articles/k/kr/kreditnaya-politika-ao-alfa-bank.html', raw_text='Чтобы стать клиентом банка и получить одобрение, необходимо соответствовать ряду формальных и неформальных критериев.Возрастной цензявляется первичным фильтром: кредитование доступно гражданам РФ от 21 года до 70 лет на момент окончания действия договора. Это стандартная практика, направленная на работу с экономически активным населением.'), ParsedRule(source_url='https://alfa-gid.vercel.app/articles/k/kr/kreditnaya-politika-ao-alfa

## 6. Сравнение: ручной сбор vs автоматический

У тебя в `policy.json` — 50 правил, собранных руками. Проверим, сколько из них парсер нашёл бы автоматически.

Не сравнивай строки буквально — сравнивай по смыслу. Если правило из `policy.json` говорит «минимальный возраст 21», и в списке кандидатов есть текст «возраст от 21 года», это попадание.

In [ ]:
print(f"Ручных правил в policy.json: {len(policy)}")
print("Кандидатов от парсера: [твоё число]")

# Запиши вывод: сколько правил ты бы пропустил, если бы собирал только парсером.

## 7. Артефакт тетради

К концу у тебя должно быть:

1. **`src/underwriting/parsers.py`** — модуль с тремя функциями и dataclass `ParsedRule`.
2. **`tests/test_parsers.py`** — 5 тестов, все зелёные.
3. **`data/parsed_candidates.json`** — список кандидатов с реальных сайтов.
4. **`~/.ssl/russian_ca_bundle.pem`** — объединённая связка сертификатов.
5. **Коммит** — с сообщением `feat: html parsers for credit policy sources`.

## 8. Что закрыто по чеклисту

- **№11 (парсеры сайтов и данных):** полностью. Ты написал `fetch_html`, `extract_rules_from_html` и обёртку над списком URL.
- **№4 (работа с API и источниками):** HTTP-запросы через `requests`, работа с реальными внешними источниками.
- **№15 (SQL, HTTP, REST):** HTTP и клиент-серверная архитектура — отработано на практике.
- **№9 (логирование, обработка ошибок):** `logger.warning`, `try/except` на уровне запроса.
- **№18 (отладка):** разобрался с SSL-сертификатом — реальная проблема, с которой сталкиваются разработчики.

**Что дальше:** тетрадь 1.2 про PDF — генерация заявок через `reportlab` и чтение через `pdfplumber`.